CROSS-CHECK R -- ANALISIS PANEL DINAMIS GMM PDRB HIJAU (2010-2025)

TUJUAN: skrip ini TIDAK menggantikan hasil utama Python (pydynpd) di
FullPanel3.ipynb. Tujuannya untuk:
  1. LLC & IPS
  2. Pesaran CD (plm::pcdtest)
  3. Windmeijer-corrected SE (pdynmc std.err="corrected")
  4. Difference-in-Hansen utk System GMM (pydynpd tidak native)

In [ ]:
R.version.string

NameError: name 'R' is not defined

In [ ]:
# ============================================================
# TAHAP 0 -- INSTALL & MOUNT DRIVE
# ============================================================
install.packages(c("plm", "pdynmc", "punitroots", "car", "dplyr"), repos = "https://cloud.r-project.org")

library(plm)
library(pdynmc)
library(punitroots)   # LLC, IPS, Pesaran CD/CIPS generasi-2
library(car)
library(dplyr)

FOLDER_OUTPUT <- "/content/drive/MyDrive/DatabaseSkripsi/Outputs"

NameError: name 'install' is not defined

In [ ]:
# ============================================================
# TAHAP 1 -- BACA PANEL
# ============================================================
df <- read.csv(file.path(FOLDER_OUTPUT, "panel_final_gmm_tahap1.csv"),
                stringsAsFactors = FALSE)

cat("Jumlah baris:", nrow(df), "| Jumlah provinsi:", length(unique(df$PROVINSI)), "\n")
cat("Rentang tahun:", min(df$year), "-", max(df$year), "\n")

cat("\n[Sanity-check kolom kunci -- bandingkan dgn output describe() Python]\n")
print(summary(df[, c("ln_pdrb_hijau", "growth_pdrb_hijau", "ln_pmtb_per_pekerja",
                      "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean")]))

panel <- pdata.frame(df, index = c("PROVINSI", "year"))

In [ ]:
# ============================================================
# TAHAP 2 -- UJI AKAR UNIT PANEL: LLC & IPS
# ============================================================
cat("\n", strrep("=", 70), "\n[R] TAHAP III -- LLC & IPS (LEVEL & GROWTH)\n", strrep("=", 70), "\n", sep = "")

# LLC & IPS BUTUH panel BALANCED (T seragam) -- Kalimantan Utara py T lebih
# pendek (NaN di tahun2 sblm 2013 utk beberapa kolom). Jalankan di DUA
# kondisi spy transparan, konsisten dgn catatan notebook Python ("Fisher/
# Maddala-Wu lebih cocok utk unbalanced, tapi WAJIB dilengkapi LLC/IPS"):

jalankan_llc_ips <- function(data_vec_list, label) {
  cat("\n---", label, "---\n")
  tryCatch({
    llc <- levinLin(data_vec_list, lags = "SIC")
    cat("LLC  : stat =", llc$statistic, "| p =", llc$p.value, "\n")
  }, error = function(e) cat("LLC  : GAGAL --", conditionMessage(e), "(kemungkinan panel tak balanced)\n"))
  tryCatch({
    ips <- ips.test(data_vec_list, lag.max = 2, lshort = TRUE)
    cat("IPS  : stat =", ips$statistic, "| p =", ips$p.value, "\n")
  }, error = function(e) cat("IPS  : GAGAL --", conditionMessage(e), "(kemungkinan panel tak balanced)\n"))
}

# 2a. Sampel LENGKAP (34 provinsi, termasuk Kalimantan Utara -- unbalanced)
for (kolom in c("ln_pdrb_hijau", "growth_pdrb_hijau")) {
  mat <- pdata.frame(df[, c("PROVINSI", "year", kolom)], index = c("PROVINSI", "year"))
  jalankan_llc_ips(mat[[kolom]], paste0(kolom, " -- 34 provinsi (unbalanced)"))
}

# 2b. Sampel TANPA Kalimantan Utara (33 provinsi --
#     balanced, hasil ini yang lebih valid utk LLC yg butuh T seragam)
df_33 <- df[df$PROVINSI != "KALIMANTAN UTARA", ]
for (kolom in c("ln_pdrb_hijau", "growth_pdrb_hijau")) {
  mat <- pdata.frame(df_33[, c("PROVINSI", "year", kolom)], index = c("PROVINSI", "year"))
  jalankan_llc_ips(mat[[kolom]], paste0(kolom, " -- 33 provinsi tanpa Kaltara"))
}

cat("\n[BANDINGKAN] Kesimpulan LLC/IPS (stasioner/tidak per level & growth) HARUS\n",
    "searah dgn Fisher/Maddala-Wu Python (Cell 12): level TIDAK stasioner,\n",
    "growth STASIONER. Kalau bertentangan, itu TEMUAN yang wajib dibahas\n",
    "eksplisit di Bab IV (bukan dipilih yang mendukung kesimpulan saja).\n", sep = "")

In [ ]:
# ============================================================
# TAHAP 3 -- PESARAN CD
# ============================================================
cat("\n", strrep("=", 70), "\n[R] TAHAP III -- PESARAN CD (pembanding thd Cell 12 Python)\n", strrep("=", 70), "\n", sep = "")

model_cd <- growth_pdrb_hijau ~ ln_pmtb_per_pekerja + IPM + ln_pdrb_kapita + ndvi_mean + lst_mean
cd_test <- pcdtest(model_cd, data = panel, model = "within")
cat("Pesaran CD (plm::pcdtest): stat =", cd_test$statistic, "| p =", cd_test$p.value, "\n")
cat("[BANDINGKAN] Python Cell 12 melaporkan CD =", "30.3899, p=0.0000 (SIGNIFIKAN).\n",
    "Angka R TIDAK PERLU identik (beda implementasi/koreksi kecil), TAPI\n",
    "KESIMPULAN (signifikan/tidak) WAJIB sama arah. Kalau berbeda arah,\n",
    "STOP -- ada bug di salah satu implementasi, jangan lanjut ke Tahap VI.\n", sep = "")

In [ ]:
# ============================================================
# TAHAP 4 -- MODEL PANEL STATIS (pembanding Chow/BP-LM/Hausman)
# ============================================================
cat("\n", strrep("=", 70), "\n[R] TAHAP IV -- MODEL PANEL STATIS (pembanding thd Cell 13 Python)\n", strrep("=", 70), "\n", sep = "")

model_statis <- ln_pdrb_hijau ~ ln_pmtb_per_pekerja + IPM + ln_pdrb_kapita
pooled_s <- plm(model_statis, data = panel, model = "pooling")
fe_s     <- plm(model_statis, data = panel, model = "within")
re_s     <- plm(model_statis, data = panel, model = "random")

cat("\n[IV.1] Chow (F-test, FE vs Pooled):\n"); print(pFtest(fe_s, pooled_s))
cat("\n[IV.2] Breusch-Pagan LM (Pooled vs RE):\n"); print(plmtest(pooled_s, type = "bp"))
cat("\n[IV.3] Hausman (FE vs RE):\n"); print(phtest(fe_s, re_s))
cat("[BANDINGKAN] Python Cell 13: Chow p=0.000000 (FE), LM p=0.917531 (Pooled),\n",
    "Hausman p=0.934657 (RE konsisten). CATATAN: Cell 13 Python memakai\n",
    "'X_s.assign(const=1)' yang SAMA dipakai utk fe_s -- pola yg SUDAH terbukti\n",
    "berisiko AbsorbingEffectError di Tahap III (Pesaran CD) & sudah diperbaiki\n",
    "di sana, TAPI belum di Tahap IV. Kalau angka R jauh berbeda dari Python\n",
    "di atas, itu indikasi kuat const-redundant tsb memang mendistorsi hasil\n",
    "Python Tahap IV -- segera terapkan perbaikan yang sama sperti Tahap III.\n", sep = "")

In [ ]:
# ============================================================
# TAHAP 5 -- VIF
# ============================================================
cat("\n", strrep("=", 70), "\n[R] VIF (pembanding thd Cell 6 Python)\n", strrep("=", 70), "\n", sep = "")
vif_model <- lm(growth_pdrb_hijau ~ ln_pmtb_per_pekerja + IPM + ln_pdrb_kapita + ndvi_mean + lst_mean,
                 data = df)
print(car::vif(vif_model))
cat("[BANDINGKAN] Python Cell 6: ln_pmtb=4.67, IPM=1.46, ln_pdrb_kapita=4.66,\n",
    "ndvi=4.84, lst=4.69 (semua < 10, aman).\n", sep = "")

In [ ]:
# ============================================================
# TAHAP 6 -- GMM DINAMIS via pdynmc
# ============================================================
cat("\n", strrep("=", 70), "\n[R] TAHAP VI-VII -- GMM DINAMIS (pdynmc) -- CROSS-CHECK pydynpd\n", strrep("=", 70), "\n", sep = "")

df_gmm <- df %>%
  filter(!is.na(growth_pdrb_hijau), !is.na(ln_pmtb_per_pekerja), !is.na(IPM),
         !is.na(ln_pdrb_kapita), !is.na(ndvi_mean), !is.na(lst_mean)) %>%
  arrange(PROVINSI, year)
df_gmm$PROVINSI <- as.factor(df_gmm$PROVINSI)

jalankan_pdynmc <- function(data_input, use_level, label) {
  cat("\n---", label, "---\n")
  cat("Observasi:", nrow(data_input), "| Provinsi:", length(unique(data_input$PROVINSI)), "\n")
  m <- tryCatch({
    pdynmc(
      dat = data_input, varname.i = "PROVINSI", varname.t = "year",
      use.mc.diff = TRUE, use.mc.lev = use_level,
      include.y = TRUE, varname.y = "growth_pdrb_hijau", lagTerms.y = 1,
      fur.con = TRUE, fur.con.diff = TRUE, fur.con.lev = use_level,
      varname.reg.fur = c("ln_pmtb_per_pekerja", "IPM", "ln_pdrb_kapita", "ndvi_mean", "lst_mean"),
      lagTerms.reg.fur = c(0, 0, 0, 0, 0),
      include.dum = TRUE, dum.diff = TRUE, dum.lev = use_level,
      w.mat = "iid.err", std.err = "corrected",   # <- koreksi Windmeijer (2005)
      estimation = "twostep",
      opt.meth = "none"
    )
  }, error = function(e) { cat("GAGAL:", conditionMessage(e), "\n"); NULL })
  if (!is.null(m)) {
    print(summary(m))
    cat("\nAR(1):\n"); print(mtest.pdynmc(m, order = 1))
    cat("AR(2):\n"); print(mtest.pdynmc(m, order = 2))
    cat("Hansen J (overid):\n"); print(jtest.pdynmc(m))
    if (use_level) {
      cat("Diff-in-Hansen (subset instrumen level-equation, khusus System GMM):\n")
      tryCatch(print(jtest.pdynmc(m, subset = "diff")),
                error = function(e) cat("  (fungsi diff-in-Hansen versi pdynmc Anda mungkin beda nama/argumen --\n",
                                          "   cek dokumentasi 'jtest.pdynmc' / 'lm_diff' versi terpasang)\n"))
    }
  }
  m
}

model_diff_r <- jalankan_pdynmc(df_gmm, use_level = FALSE, "DIFFERENCE GMM (R, pdynmc)")
model_sys_r  <- jalankan_pdynmc(df_gmm, use_level = TRUE,  "SYSTEM GMM (R, pdynmc)")

cat("\n[BANDINGKAN WAJIB thd Python Cell 15-17]\n",
    "  Difference GMM Python : koef lag = -0.2749 (p=0.0000), Hansen p=0.2452, AR2 p=0.2300\n",
    "  System GMM Python     : koef lag = -0.2943 (p=0.0005), Hansen p=0.2692, AR2 p=0.2790\n",
    "  SE two-step Python (deteksi kasar Cell 17): Diff rasio=0.65 (agak besar penurunannya --\n",
    "    curigai belum terkoreksi Windmeijer), System rasio=0.96 (wajar).\n",
    "  -> R std.err='corrected' memberi SE yang SUDAH pasti terkoreksi Windmeijer.\n",
    "     Kalau SE dan koefisien lag dari R cukup dekat dgn Python (beda kecil,\n",
    "     wajar krn beda software), itu BUKTI robustness kuat utk sidang.\n",
    "     Kalau beda BESAR khususnya pada Difference GMM (yang rasio SE-nya\n",
    "     mencurigakan di Python), pertimbangkan melaporkan hasil R sbg hasil\n",
    "     utama utk SE (koefisien tetap dari pydynpd, SE dari pdynmc, sebutkan\n",
    "     eksplisit di catatan kaki tabel -- praktik umum saat cross-check R\n",
    "     mengungkap perbedaan material).\n", sep = "")

In [ ]:
# ============================================================
# TAHAP 7 -- EKSPOR
# ============================================================
cat("\n", strrep("=", 70), "\n[R] EKSPOR HASIL CROSS-CHECK\n", strrep("=", 70), "\n", sep = "")

ekstrak_ringkas <- function(model_obj, label) {
  if (is.null(model_obj)) return(data.frame(model = label, status = "GAGAL"))
  koef <- coef(model_obj)
  lag_nama <- grep("^growth_pdrb_hijau|lag", names(koef), value = TRUE, ignore.case = TRUE)[1]
  data.frame(model = label, koef_lag = koef[lag_nama], status = "OK")
}

rekap_r <- rbind(
  ekstrak_ringkas(model_diff_r, "Difference GMM (R)"),
  ekstrak_ringkas(model_sys_r, "System GMM (R)")
)
write.csv(rekap_r, file.path(FOLDER_OUTPUT, "crosscheck_R_ringkasan.csv"), row.names = FALSE)
cat("Rekap ringkas disimpan ->", file.path(FOLDER_OUTPUT, "crosscheck_R_ringkasan.csv"), "\n")
cat("\nCATATAN AKHIR: simpan JUGA seluruh output console di atas (screenshot/copy)\n",
    "sbg lampiran cross-check R -- lebih lengkap dari file CSV ringkas ini,\n",
    "dan itu yang dirujuk 'alur-gmm-panel-dinamis.md' Bagian E sbg bukti\n",
    "robustness tambahan untuk sidang.\n", sep = "")